# Query a Fabric Ontology through MCP

Connect directly to the Fabric IQ ontology MCP endpoint, inspect the tools it exposes, and call one of those tools with a natural-language question. The notebook signs you in to the Fabric tenant through the browser, and your account needs access to the Fabric workspace.

## Step 1: Load configuration and create an MCP session helper

Load the Fabric tenant and ontology MCP endpoint written to the project `.env` file during provisioning. The HTTP client follows redirects because Fabric routes MCP requests through its service endpoint.

In [5]:
import os
from contextlib import asynccontextmanager

import httpx
from azure.identity import InteractiveBrowserCredential
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

load_dotenv(override=True)

FABRIC_TENANT_ID = os.environ["FABRIC_TENANT_ID"]
FABRIC_ONTOLOGY_MCP_URL = os.environ["FABRIC_ONTOLOGY_MCP_URL"]
FABRIC_SCOPE = "https://api.fabric.microsoft.com/.default"

credential = InteractiveBrowserCredential(tenant_id=FABRIC_TENANT_ID)


@asynccontextmanager
async def open_ontology_session():
    token = credential.get_token(FABRIC_SCOPE)

    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {token.token}"},
        follow_redirects=True,
        timeout=httpx.Timeout(30, read=300),
    ) as http_client:
        async with streamable_http_client(FABRIC_ONTOLOGY_MCP_URL, http_client=http_client,) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session

print("Fabric ontology configuration loaded")

Fabric ontology configuration loaded


## Step 2: List the available tools

Initialize an authenticated MCP session and inspect every tool exposed by the Fabric ontology, including each tool's input schema.

In [7]:
from rich.console import Console

console = Console()

async with open_ontology_session() as session:
    tools_response = await session.list_tools()

ontology_tools = tools_response.tools

for tool in ontology_tools:
    console.print(f"[bold cyan]Tool:[/] {tool.name}")
    if tool.description:
        console.print(f"[bold]Description:[/] {tool.description}")
    console.print("[bold]Input schema:[/]")
    console.print_json(data=tool.inputSchema)
    console.print()

Tool: ask_ontology

Description: Answers day-to-day business questions using an established Fabric ontology and its bound data, or 
generates grounded, validated, runnable code. For code requests, include the required `queryLanguage` and execution
target inside the `request`. The requested artifact can be native KQL, SQL, DAX, cloud-runnable Python, a DSL 
definition, or Ontology or Eventhouse-runnable KQL that executes and joins KQL, SQL, and DAX sources. The tool 
performs read-only operations and returns either an answer with supporting evidence or validated runnable code with
grounding and validation details.

Input schema:

{
  "type": "object",
  "properties": {
    "request": {
      "type": "string",
      "description": "The business question or requested artifact. Include the relevant scope, filters, time range, and desired output, such as an answer, data, or reusable code. For code requests, include `queryLanguage` and the intended execution target in the prompt.",
      "minLength": 3,
      "maxLength": 10000
    }
  },
  "required": [
    "request"
  ]
}

Tool: list_ontology_rules

...

{
  "type": "object",
  "properties": {
    "entityNames": {
      "type": "array",
      "description": "Optional list of entity names to filter by. Only rules that reference at least one of the named entities are returned. Matching is exact and case-insensitive; use the entity's full composite name (for example 'plant#Technician') for entities outside the default namespace. Only a rule's referencedEntities are considered — a rule that reaches an entity solely through a referencedRelationships endpoint is not matched. An empty list is not accepted and every name must be non-empty; omit this parameter to return all rules.",
      "items": {
        "type": "string"
      },
      "minItems": 1,
      "maxItems": 100
    }
  },
  "required": []
}

Tool: list_ontology_entities

Description: Lists the entities defined in the ontology, including each entity's properties, keys, backing tables, 
annotations, and resource links with resolved item identities when available. Prefer this tool for exploring 
ontology entities; if it returns a not-supported error, retry with list_ontology_entity_types.

Input schema:

{
  "type": "object",
  "properties": {
    "entityName": {
      "type": "string",
      "description": "Optional exact name of an entity to filter by. May return more than one match because entity names can repeat across namespaces. An empty value is not accepted; omit this parameter to list all entities.",
      "minLength": 1,
      "maxLength": 128,
      "pattern": "^[a-zA-Z][a-zA-Z0-9_-]{0,127}$"
    }
  },
  "required": []
}

## Step 3: Query the Fabric ontology

Call the ontology's `ask_ontology` tool with a natural-language business question. The tool answers from the ontology's bound lakehouse data and returns the answer with supporting evidence. Change `question` to explore other product, supplier, store, and inventory questions.

Answers typically take 30 to 60 seconds. If a question takes longer than about a minute, Fabric can return an empty response, which surfaces as `Error parsing JSON response` followed by a timeout. If that happens, re-run the cell.

In [4]:
from datetime import timedelta

from rich.markdown import Markdown

question = "Which suppliers supply the Standard Outlet 15-Amp, and what are their lead times?"

async with open_ontology_session() as session:
    result = await session.call_tool(
        "ask_ontology",
        {"request": question},
        read_timeout_seconds=timedelta(seconds=120),
    )

answer = "".join(block.text for block in result.content if block.type == "text")
console.print(Markdown(answer))

The Standard Outlet 15-Amp is supplied by these suppliers:                                                         

                                     
 Supplier                  Lead time 
 ─────────────────────────────────── 
 Rainier Building Supply     14 days 
 Columbia Plumbing Supply    10 days 
                                     

If you want, I can also show the product SKU or sort these by fastest lead time.